In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

for folder_name in ["data", "checkpoints", "outputs"]:
    folder_path = drive_root / folder_name

    if not folder_path.is_dir():
        raise RuntimeError(f"Lỗi: Chưa có thư mục Drive: {folder_path}")

print("Đã kết nối Drive và tìm thấy đủ thư mục dự án.")

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "1ab81cf320b185c1213726142ed55ef5c0a56e24"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError("Lỗi: Repository còn thay đổi. Hãy dùng runtime mới.")

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

commit_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

git_commit = commit_result.stdout.strip()

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit nền:", git_commit)
print("Thư mục làm việc:", Path.cwd())

In [ ]:
%pip uninstall -y diffusers
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

In [ ]:
import importlib
import json
import platform
from importlib.metadata import version

packages = {
    "torch": "torch",
    "transformers": "transformers",
    "datasets": "datasets",
    "sentencepiece": "sentencepiece",
    "rouge-score": "rouge_score",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "gradio": "gradio",
    "numpy": "numpy",
}

library_versions = {}

for package_name in packages:
    module_name = packages[package_name]
    importlib.import_module(module_name)
    library_versions[package_name] = version(package_name)

result = {
    "python_version": platform.python_version(),
    "library_versions": library_versions,
    "import_check_passed": True,
}

print(json.dumps(result, ensure_ascii=False, indent=2))

In [ ]:
import json
import platform
import torch
from datetime import datetime, timezone

environment = {
    "run_id": "colab_environment_check",
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "pytorch_version": str(torch.__version__),
    "pytorch_cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpus": [],
    "gpu_check_passed": False,
    "diagnostics": [],
}

if environment["cuda_available"]:
    for index in range(torch.cuda.device_count()):
        gpu = torch.cuda.get_device_properties(index)

        gpu_info = {
            "index": index,
            "name": gpu.name,
            "total_vram_gib": round(gpu.total_memory / 1024**3, 2),
        }

        environment["gpus"].append(gpu_info)

    if len(environment["gpus"]) > 0:
        environment["gpu_check_passed"] = True
else:
    environment["diagnostics"].append(
        "Chưa truy cập được GPU. Hãy kiểm tra loại runtime."
    )

print(json.dumps(environment, ensure_ascii=False, indent=2))

if not environment["gpu_check_passed"]:
    raise RuntimeError("Lỗi: Kiểm tra môi trường GPU chưa đạt.")

In [ ]:
import json

if "environment" not in globals():
    raise RuntimeError("Lỗi: Chưa có dữ liệu môi trường.")

if "drive_root" not in globals():
    raise RuntimeError("Lỗi: Chưa kết nối Drive.")

if "git_commit" not in globals():
    raise RuntimeError("Lỗi: Chưa xác định commit đang sử dụng.")

if "library_versions" not in globals():
    raise RuntimeError("Lỗi: Chưa kiểm tra phiên bản thư viện.")

environment["git_commit"] = git_commit
environment["library_versions"] = library_versions

report_path = drive_root / "outputs" / environment["run_id"] / "config.json"
report_path.parent.mkdir(parents=True, exist_ok=True)

report_path.write_text(
    json.dumps(environment, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Đã lưu hồ sơ trực tiếp trên Drive:", report_path)

In [ ]:
import importlib
from google.colab import files

source_paths = [
    "configs/train.json",
    "src/data.py",
    "src/tokenization.py",
    "src/train.py",
]

for relative_path in source_paths:
    print("Chọn file:", relative_path)
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise RuntimeError("Lỗi: Mỗi lượt chỉ chọn đúng một file.")

    for file_name in uploaded:
        uploaded_content = uploaded[file_name]

    target_path = repo_dir / relative_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    target_path.write_bytes(uploaded_content)

    print("Đã cập nhật:", target_path)

import src.data
import src.tokenization
import src.train

importlib.reload(src.data)
importlib.reload(src.tokenization)
importlib.reload(src.train)

In [ ]:
import json
from src.utils import load_train_config, set_seed

train_config = load_train_config()
set_seed(train_config["seed"])

tokenized_splits, tokenizer = src.tokenization.tokenize_train_validation(
    drive_root,
)

data_loaders = src.tokenization.create_train_validation_loaders(
    tokenized_splits,
    tokenizer,
)

forward_batch = next(iter(data_loaders["train"]))

batch_shapes = {}

for field in ["input_ids", "attention_mask", "labels"]:
    batch_shapes[field] = list(forward_batch["model_inputs"][field].shape)

print(json.dumps(
    {
        "sample_ids": forward_batch["sample_ids"],
        "batch_size": train_config["batch_size"],
        "shapes": batch_shapes,
    },
    ensure_ascii=False,
    indent=2,
))

In [ ]:
model = src.train.load_vit5_model()

forward_report = src.train.check_forward_batch(
    model,
    forward_batch,
)

forward_report["model_id"] = train_config["model"]["id"]
forward_report["model_revision"] = train_config["model"]["revision"]
forward_report["max_source_length"] = train_config["max_source_length"]
forward_report["max_target_length"] = train_config["max_target_length"]
forward_report["task_prefix"] = train_config["task_prefix"]
forward_report["source_mode"] = "local files uploaded for pre-commit check"
forward_report["test_content_read"] = False

print(json.dumps(
    forward_report,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
import importlib
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("Lỗi: Hãy chọn đúng một file src/train.py.")

for file_name in uploaded:
    uploaded_content = uploaded[file_name]

target_path = repo_dir / "src" / "train.py"
target_path.write_bytes(uploaded_content)

import src.train

importlib.reload(src.train)

print("Đã cập nhật src/train.py trong runtime.")

In [ ]:
import json

optimizer = src.train.create_optimizer(model)

optimization_report = src.train.check_optimization_step(
    model,
    forward_batch,
    optimizer,
)

optimization_report["model_id"] = train_config["model"]["id"]
optimization_report["model_revision"] = train_config["model"]["revision"]
optimization_report["max_source_length"] = train_config["max_source_length"]
optimization_report["max_target_length"] = train_config["max_target_length"]
optimization_report["learning_rate"] = train_config["learning_rate"]
optimization_report["weight_decay"] = train_config["weight_decay"]
optimization_report["source_mode"] = "local files uploaded for pre-commit check"

print(json.dumps(
    optimization_report,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
import gc
import torch
from src.utils import set_seed

small_sample_batches = src.train.create_small_sample_batches(
    tokenized_splits["train"],
    tokenizer,
    seed=train_config["seed"],
)

if "optimizer" in globals():
    del optimizer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

set_seed(train_config["seed"])

model = src.train.load_vit5_model()
optimizer = src.train.create_optimizer(model)

print("Đã chuẩn bị 16 mẫu train cố định.")
print("Đã tải lại ViT5-Base và tạo optimizer mới.")

In [ ]:
import json

small_sample_report = src.train.run_small_sample_check(
    model,
    small_sample_batches,
    optimizer,
)

small_sample_report["model_id"] = train_config["model"]["id"]
small_sample_report["model_revision"] = train_config["model"]["revision"]
small_sample_report["seed"] = train_config["seed"]
small_sample_report["precision"] = train_config["precision"]
small_sample_report["max_source_length"] = train_config["max_source_length"]
small_sample_report["max_target_length"] = train_config["max_target_length"]
small_sample_report["learning_rate"] = train_config["learning_rate"]
small_sample_report["weight_decay"] = train_config["weight_decay"]
small_sample_report["initialization"] = "fresh pretrained model and optimizer"
small_sample_report["source_mode"] = "local files uploaded for pre-commit check"

print(json.dumps(
    small_sample_report,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
model_origin = "existing runtime model"

if "model" not in globals():
    model = src.train.load_vit5_model()
    model_origin = "fresh pretrained ViT5-Base"

print("Điểm khởi đầu:", model_origin)
print("Số mẫu validation:", len(tokenized_splits["validation"]))

In [ ]:
import json

expected_validation_ids = []

for sample in tokenized_splits["validation"]:
    expected_validation_ids.append(sample["id"])

validation_report = src.train.validate_model(
    model,
    data_loaders["validation"],
    expected_validation_ids,
)

validation_report["model_origin"] = model_origin
validation_report["model_id"] = train_config["model"]["id"]
validation_report["model_revision"] = train_config["model"]["revision"]
validation_report["precision"] = train_config["precision"]
validation_report["batch_size"] = train_config["batch_size"]
validation_report["max_source_length"] = train_config["max_source_length"]
validation_report["max_target_length"] = train_config["max_target_length"]
validation_report["source_mode"] = "local files uploaded for pre-commit check"

print(json.dumps(
    validation_report,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
import gc
import torch
from datetime import datetime, timezone
from src.utils import set_seed

if "optimizer" in globals():
    del optimizer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

set_seed(train_config["seed"])

model = src.train.load_vit5_model()
optimizer = src.train.create_optimizer(model)

checkpoint_batch = forward_batch

probe_text = tokenizer.decode(
    checkpoint_batch["model_inputs"]["input_ids"][0].tolist(),
    skip_special_tokens=True,
)

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
run_id = "checkpoint_roundtrip_check_" + timestamp
checkpoint_dir = drive_root / "checkpoints" / run_id / "last"

print("Checkpoint sẽ lưu tại:", checkpoint_dir)
print("Mẫu train:", checkpoint_batch["sample_ids"])

In [ ]:
first_step_report = src.train.check_optimization_step(
    model,
    checkpoint_batch,
    optimizer,
)

probe_before = src.train.generate_checkpoint_probe(
    model,
    tokenizer,
    probe_text,
)

checkpoint_progress = {
    "completed_epochs": 0,
    "global_step": 1,
}

src.train.save_training_checkpoint(
    model,
    tokenizer,
    optimizer,
    checkpoint_dir,
    checkpoint_progress,
)

print("Đã lưu model, tokenizer, optimizer, tiến độ và trạng thái ngẫu nhiên.")
print("Tiến độ đã lưu:", checkpoint_progress)

In [ ]:
del optimizer
del model

gc.collect()
torch.cuda.empty_cache()

model, tokenizer, optimizer, progress, restoration_report = (
    src.train.load_training_checkpoint(checkpoint_dir)
)

print(json.dumps(
    restoration_report,
    ensure_ascii=False,
    indent=2,
))

In [ ]:
probe_after = src.train.generate_checkpoint_probe(
    model,
    tokenizer,
    probe_text,
)

for field in ["input_ids", "attention_mask", "generated_ids", "generated_text"]:
    if probe_before[field] != probe_after[field]:
        raise RuntimeError("Lỗi: Kết quả sau tải lại không khớp: " + field)

if progress != checkpoint_progress:
    raise RuntimeError("Lỗi: Tiến độ phục hồi không khớp.")

if restoration_report["optimizer_step_values"] != [1]:
    raise RuntimeError("Lỗi: Optimizer chưa phục hồi đúng bước 1.")

resumed_step_report = src.train.check_optimization_step(
    model,
    checkpoint_batch,
    optimizer,
)

progress["global_step"] = progress["global_step"] + 1

optimizer_step_values = set()

for parameter_state in optimizer.state.values():
    step_value = int(parameter_state["step"].item())
    optimizer_step_values.add(step_value)

if optimizer_step_values != {2} or progress["global_step"] != 2:
    raise RuntimeError("Lỗi: Tiến độ sau bước học tiếp chưa đúng.")

checkpoint_files = []

for file_path in checkpoint_dir.iterdir():
    if file_path.is_file():
        checkpoint_files.append(file_path.name)

checkpoint_report = {
    "run_id": run_id,
    "checkpoint_dir": str(checkpoint_dir),
    "checkpoint_files": sorted(checkpoint_files),
    "sample_ids": checkpoint_batch["sample_ids"],
    "model_id": train_config["model"]["id"],
    "model_revision": train_config["model"]["revision"],
    "restoration": restoration_report,
    "tokenizer_check_passed": True,
    "generated_ids_check_passed": True,
    "generated_text_check_passed": True,
    "source_token_count": len(probe_before["input_ids"]),
    "generated_ids_before": probe_before["generated_ids"],
    "generated_ids_after": probe_after["generated_ids"],
    "generated_text": probe_after["generated_text"],
    "generation_config": probe_after["generation_config"],
    "saved_progress": checkpoint_progress,
    "progress_after_resumed_step": progress,
    "optimizer_step_values_after_resume": sorted(optimizer_step_values),
    "first_step": first_step_report,
    "resumed_step": resumed_step_report,
    "test_content_read": False,
    "source_mode": "local files uploaded for pre-commit check",
}

print(json.dumps(
    checkpoint_report,
    ensure_ascii=False,
    indent=2,
))

In [11]:
import os
import importlib
from google.colab import files

source_paths = [
    "configs/train.json",
    "src/train.py",
]

for relative_path in source_paths:
    print("Chọn file:", relative_path)

    os.chdir("/content")

    try:
        uploaded = files.upload()
    finally:
        os.chdir(repo_dir)

    if len(uploaded) != 1:
        raise RuntimeError("Lỗi: Mỗi lượt chỉ chọn đúng một file.")

    for file_name in uploaded:
        uploaded_content = uploaded[file_name]

    target_path = repo_dir / relative_path
    target_path.write_bytes(uploaded_content)

    print("Đã cập nhật:", relative_path)

import src.train

importlib.reload(src.train)

Chọn file: configs/train.json


Saving train.json to train.json
Đã cập nhật: configs/train.json
Chọn file: src/train.py


Saving train.py to train (1).py
Đã cập nhật: src/train.py


<module 'src.train' from '/content/vietnamese-summarization/src/train.py'>

In [12]:
import json

run_profile = src.train.create_training_run(
    repo_dir,
    drive_root,
    allow_uncommitted=True,
)

profile_path = (
    Path(run_profile["paths"]["output_dir"]) / "config.json"
)

print("Đã lưu và đọc lại hồ sơ thành công:", profile_path)

print(json.dumps(
    run_profile,
    ensure_ascii=False,
    indent=2,
))

Đã lưu và đọc lại hồ sơ thành công: /content/drive/MyDrive/vietnamese-summarization/outputs/vit5_finetuning_20261105_171113_817490/config.json
{
  "run_id": "vit5_finetuning_20261105_171113_817490",
  "created_at_utc": "2026-11-05T17:11:13.817490+00:00",
  "run_status": "setup_only",
  "source": {
    "repo_url": "https://github.com/MD1809/vietnamese-summarization.git",
    "git_commit": "1ab81cf320b185c1213726142ed55ef5c0a56e24",
    "commit_on_fetched_origin_main": true,
    "has_uncommitted_changes": true,
    "status_porcelain": " M configs/train.json\n M src/train.py\n",
    "allow_uncommitted": true
  },
  "train_config": {
    "config_status": "initial",
    "model": {
      "id": "VietAI/vit5-base",
      "revision": "2209a38d735ede63e88f5aa52bcdc11a05a37b85"
    },
    "dataset": {
      "id": "esdurmus/wiki_lingua",
      "revision": "ea3db3510cbd34d0f8dc612419ae40e4732f3b40",
      "language": "vietnamese"
    },
    "paths": {
      "train_file": "data/processed/train.jsonl

In [ ]:
import gc
import torch
import src.tokenization
from src.utils import load_train_config, set_seed

train_config = load_train_config()

tokenized_splits, tokenizer = src.tokenization.tokenize_train_validation(
    drive_root,
)

resource_batches = src.train.create_small_sample_batches(
    tokenized_splits["train"],
    tokenizer,
    seed=train_config["seed"],
)

if "optimizer" in globals():
    del optimizer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

set_seed(train_config["seed"])

model = src.train.load_vit5_model()
optimizer = src.train.create_optimizer(model)

print("Đã chuẩn bị 16 mẫu train, ViT5-Base và AdamW mới.")

In [ ]:
import json
from pathlib import Path

resource_report = src.train.run_resource_check(
    model,
    resource_batches,
    optimizer,
    tokenizer,
)

resource_report["run_id"] = run_profile["run_id"]
resource_report["model_id"] = train_config["model"]["id"]
resource_report["model_revision"] = train_config["model"]["revision"]
resource_report["train_sample_count"] = len(tokenized_splits["train"])
resource_report["run_config_file"] = str(
    Path(run_profile["paths"]["output_dir"]) / "config.json"
)

report_path = (
    Path(run_profile["paths"]["output_dir"]) / "resource_check.json"
)

report_path.write_text(
    json.dumps(resource_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

saved_report = json.loads(report_path.read_text(encoding="utf-8"))

if saved_report != resource_report:
    raise RuntimeError("Lỗi: Báo cáo đọc lại không khớp.")

print("Đã lưu và đọc lại báo cáo:", report_path)

print(json.dumps(
    resource_report,
    ensure_ascii=False,
    indent=2,
))

In [13]:
import gc
import torch
import src.tokenization
from src.utils import load_train_config, set_seed

train_config = load_train_config()

tokenized_splits, tokenizer = src.tokenization.tokenize_train_validation(
    drive_root,
)

accumulation_batches = src.train.create_small_sample_batches(
    tokenized_splits["train"],
    tokenizer,
    seed=train_config["seed"],
    sample_count=17,
)

if "optimizer" in globals():
    del optimizer

if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

set_seed(train_config["seed"])

model = src.train.load_vit5_model()
optimizer = src.train.create_optimizer(model)

print("Đã chuẩn bị 17 mẫu train và mô hình mới.")
print("Accumulation:", train_config["gradient_accumulation_steps"])
print("Ngưỡng clipping:", train_config["max_grad_norm"])

Đã chuẩn bị 17 mẫu train và mô hình mới.
Accumulation: 8
Ngưỡng clipping: 1.0


In [14]:
import json
from pathlib import Path

accumulation_report = src.train.run_accumulation_check(
    model,
    accumulation_batches,
    optimizer,
)

accumulation_report["run_id"] = run_profile["run_id"]
accumulation_report["model_id"] = train_config["model"]["id"]
accumulation_report["model_revision"] = train_config["model"]["revision"]
accumulation_report["max_source_length"] = train_config["max_source_length"]
accumulation_report["max_target_length"] = train_config["max_target_length"]
accumulation_report["run_config_file"] = str(
    Path(run_profile["paths"]["output_dir"]) / "config.json"
)

report_path = (
    Path(run_profile["paths"]["output_dir"])
    / "gradient_accumulation_check.json"
)

report_path.write_text(
    json.dumps(accumulation_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

saved_report = json.loads(report_path.read_text(encoding="utf-8"))

if saved_report != accumulation_report:
    raise RuntimeError("Lỗi: Báo cáo đọc lại không khớp.")

print("Đã lưu và đọc lại báo cáo:", report_path)

print(json.dumps(
    accumulation_report,
    ensure_ascii=False,
    indent=2,
))

Nhóm: 1 Số batch: 8
Nhóm: 2 Số batch: 8
Nhóm: 3 Số batch: 1
Đã lưu và đọc lại báo cáo: /content/drive/MyDrive/vietnamese-summarization/outputs/vit5_finetuning_20261105_171113_817490/gradient_accumulation_check.json
{
  "sample_ids": [
    "vi_abb6471e03742c317b1d2c534e854751d71c7efa18c7549ae157305543f42830",
    "vi_1b034a9fa257884c56b2bc38742b4bc739a501f0a5e47836927b2a935e256c5b",
    "vi_07689f3fb5a1dd2b95b44f13cf725a312b8acf60aabc118a5b9160f774d2beb0",
    "vi_c5d7c82d9955defebfc96d475b50e75e5f9a6e60b76f5962d6aa86dc163c40e4",
    "vi_46fe94e1e48920a775be92e577e0d9332b174269ff6c98b1d5a8808bc10c52d2",
    "vi_3f77b55d29dca2fda46900c2d6dd45c55db81a77ce83907deec2797a9c0da9e5",
    "vi_399c1f5df69aa95f768155c0143c61d22d6269342c056c85447e32720d2b4769",
    "vi_2290ddab62514134a859a13436ca178a1ea46ddd8d08d1470f095df09859ce06",
    "vi_c5296df3a72d4b8dfadd71e3fa593b2528c0117d623394be07345e9f34d51e65",
    "vi_1889776056701de9d89884ec288a23512e6fe2ddad9edb0d927c7ef5bfcc1edb",
    "vi_b49965e